# OV5640 连续采集测试
只接管冷启动已加载的固件，**不加载bitstream、不重启、不修改启动项**。正常初始化一次、复用DMA缓冲；DMA字节长度不合格后完整复位并重新初始化，单独累计恢复次数。
按顺序运行：参数 → 环境检查 → 开始 → 结果 → 可选抽样图。需要PYNQ的root Python内核。采集时独占摄像头；不要同时运行其他capture脚本或MMIO单元。
**安全停止用Interrupt。若提示DMA停机未确认，保持kernel存活，禁止重跑/关闭/重启kernel，先给板卡断电。**

In [ ]:
from pathlib import Path
from datetime import datetime
import uuid
BASE = Path.cwd()
if not (BASE / 'continuous_capture.py').is_file():
    BASE = Path('/home/xilinx/jupyter_notebooks/ov5640_snapshot')
MANIFEST = BASE / 'release/release_manifest.json'
SECONDS = 60.0                 # 初次初始化后计时；0为仅按总尝试数限制
MAX_ATTEMPTS = 1000            # 时长或总尝试任一上限先到即停止
MAX_CONSECUTIVE_ERRORS = 5     # 有效帧会清零连续错误；累计错误另记
MAX_PIXEL_LOSS_PERCENT = 5.0  # 严格低于5%接受；等于或高于5%拒绝
MAX_SAMPLES = 3                # 最多3个RAW内存样本，结束后落盘；0为不保存
SAMPLE_EVERY = 100             # 第0次及每100次尝试，仅抽取有效帧
FRAME_TIMEOUT = 5.0
INIT_TIMEOUT = 30.0
ASSUME_COLD_BOOT = True        # 确认当前成对固件已在冷启动仅加载一次
OUTPUT = BASE / 'captures' / ('continuous_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
EXPECTED_HELPER_SHA256 = '41cd1dcd45697cac1c6c642b0bbc7fa298e05fb2e40c3f5219655dbee80358f2'
print('输出目录：', OUTPUT)
print('像素丢失门限：', MAX_PIXEL_LOSS_PERCENT, '%（严格低于门限接受）')


## 环境检查（不访问采集寄存器）
下面校验权限、Notebook/helper版本、成对文件与HWH、当前boot/download标记。文件和标记不能单独证明运行位流身份，仍需操作者确认上述冷启动前提。若版本不同，先核对文件；不要在不安全DMA状态下重启kernel。

In [ ]:
import hashlib, json, os, sys
if not hasattr(os, 'geteuid') or os.geteuid() != 0:
    raise RuntimeError('需要板端PYNQ root Python内核。请选对内核；不要在此单元加载bit。')
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
import continuous_capture as cc
assert Path(cc.__file__).resolve() == (BASE / 'continuous_capture.py').resolve(), '导入了其他目录的helper'
assert hashlib.sha256((BASE / 'continuous_capture.py').read_bytes()).hexdigest() == EXPECTED_HELPER_SHA256, 'Notebook/helper文件版本不一致'
assert cc.SOURCE_SHA256 == EXPECTED_HELPER_SHA256, '内核缓存了其他版本helper；先确认DMA安全再处理内核'
if cc.UNSAFE_SESSIONS or cc.c.RETAINED_BUFFERS:
    raise RuntimeError('上次DMA未安全停止，保持kernel存活，先断电！')
metadata, _files = cc.c.verify_manifest(MANIFEST)
boot_id = Path('/proc/sys/kernel/random/boot_id').read_text().strip()
assert Path('/run/ov5640_snapshot_download_attempted').read_text().strip() == boot_id, '当前boot/download标记不匹配'
print('权限/root、文件版本、成对固件与boot标记检查通过。')
print('当前boot：', boot_id)
print('bit SHA256：', metadata['files']['bit']['sha256'])
print('不下载bit，开始单元会再次检查互斥锁、并发进程和FCLK0。')

## 开始有界连续测试
默认最长约60秒采集，加初次初始化/清理时间。Interrupt会请求安全停止，不强杀DMA。循环中不编码PNG；限量RAW样本结束后写盘。
根据DMA实际字节数计算的像素丢失率严格低于5%的短帧计为“接受的部分帧”；等于或高于门限的帧计为坏帧并触发恢复。部分帧只保留实际接收的RAW字节，不伪造成完整720p PNG。


In [ ]:
print('开始采集：', SECONDS, '秒 /', MAX_ATTEMPTS, '次尝试；停止请用Interrupt。', flush=True)
result = cc.run_capture(OUTPUT, MANIFEST, assume_cold_boot=ASSUME_COLD_BOOT,
                        seconds=SECONDS, max_attempts=MAX_ATTEMPTS,
                        max_errors=MAX_CONSECUTIVE_ERRORS, max_samples=MAX_SAMPLES,
                        sample_every=SAMPLE_EVERY, frame_timeout=FRAME_TIMEOUT,
                        init_timeout=INIT_TIMEOUT,
                        pixel_loss_threshold_percent=MAX_PIXEL_LOSS_PERCENT)
print('已结束：', result['stop_reason'])
if result['unsafe_dma']:
    print('危险：DMA停机未确认。保留当前kernel，禁止重跑/重启/关闭，先断电！')
else:
    print('DMA安全停机：', result['dma_shutdown_confirmed'], '相机复位：', result.get('camera_reset_after_capture', False))


## 结果
`loss_percent = lost_pixels / 921600 × 100`。严格小于配置门限的部分帧计入有效帧；恰好5%及以上仍按坏帧处理。`effective_fps`包含坏帧恢复、轮询、缓冲操作和逐帧日志开销，排除初次初始化、最终清理、RAW落盘及PNG转换；不是传感器原生帧率或显示刷新率。逐次日志保存在attempts.jsonl。

In [ ]:
result = json.loads((OUTPUT / 'results.json').read_text())
keys = ['stop_reason', 'attempts', 'valid_frames', 'accepted_loss_frames',
        'bad_frames', 'timeouts', 'dma_errors',
        'errors', 'consecutive_errors', 'max_consecutive_errors_observed',
        'aborted_attempts', 'initializations', 'recoveries', 'initialization_failures',
        'pixel_loss_threshold_percent', 'total_missing_pixels', 'max_loss_percent',
        'capture_seconds', 'effective_fps', 'total_seconds', 'completed_without_errors',
        'dma_shutdown_confirmed', 'unsafe_dma', 'error']
print(json.dumps({k: result.get(k) for k in keys}, indent=2, ensure_ascii=False))
print('逐帧丢失信息：')
rows = [json.loads(line) for line in (OUTPUT / 'attempts.jsonl').read_text().splitlines()]
for row in rows:
    metrics = row if 'loss_percent' in row else row.get('diagnostics', {})
    if 'loss_percent' in metrics:
        print(json.dumps({k: row.get(k) if k in row else metrics.get(k)
                          for k in ('index', 'outcome', 'received_pixels', 'lost_pixels', 'loss_percent', 'threshold_percent')},
                         ensure_ascii=False))
print('RAW样本数量：', len(result['samples']))
print('部分帧样本数量（仅RAW）：', sum(not s.get('image_safe', False) for s in result['samples']))
print('证据目录：', OUTPUT)


## 可选：结束后转换抽样图
确认DMA安全后才运行。每张720p PNG在板端可能耗时约20秒，完全不计入上面的有效FPS。本单元只转换已保存且尺寸完整、无几何异常的RAW；低于5%门限的部分帧只保留RAW，不生成误导性PNG；不覆盖已有PNG。

In [ ]:
png_paths = cc.convert_samples(OUTPUT)
print('转换完成：', [p.name for p in png_paths])


In [ ]:
from IPython.display import Image, display
for path in sorted(OUTPUT.glob('sample_*.png'))[:MAX_SAMPLES]:
    print(path.name)
    display(Image(filename=str(path), width=960))